In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.data_loader import load_heart_disease
from src.preprocessing import encode_source, drop_ca, binarize_num, preprocess

df_raw = load_heart_disease()
print(f"Raw shape: {df_raw.shape}")
print(df_raw.dtypes)

Raw shape: (920, 15)
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
source          str
dtype: object


## Step 1 — Encode source

Replace the `source` string column with a numeric `source_code` (1 = cleveland, 2 = hungarian, 3 = long_beach_va, 4 = switzerland).

In [2]:
df = encode_source(df_raw)
print(df[["source_code"]].value_counts().sort_index())
df.head(5)

source_code
1              303
2              294
3              200
4              123
Name: count, dtype: int64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,source_code
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,1
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,1
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,1


## Step 2 — Drop `ca`

`ca` has too many missing values across sources and is excluded from the processed dataset.

In [3]:
df = drop_ca(df)
print(f"Columns after dropping ca: {list(df.columns)}")

Columns after dropping ca: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'thal', 'num', 'source_code']


## Step 3 — Binarize target (`num`)

Convert `num` (0-4 severity) to binary: 0 = no disease, 1 = disease present. `num` is moved to the last column.

In [5]:
df = binarize_num(df)
print(df["num"].value_counts().sort_index())
print(f"Column order: {list(df.columns)}")
df.head(5)

num
0    411
1    509
Name: count, dtype: int64
Column order: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'thal', 'source_code', 'num']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,thal,source_code,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,6.0,1,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,1,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,3.0,1,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,3.0,1,0


## Save processed dataset

In [6]:
out_path = "../data/processed/heart_disease_processed.csv"
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.describe()

Saved to ../data/processed/heart_disease_processed.csv
Shape: 920 rows x 14 columns


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,thal,source_code,num
count,920.000000,920.000000,920.000000,861.000000,890.000000,830.000000,918.000000,865.000000,865.000000,858.000000,611.000000,434.000000,920.000000,920.000000
mean,53.510870,0.789130,3.250000,132.132404,199.130337,0.166265,0.604575,137.545665,0.389595,0.878788,1.770867,5.087558,2.155435,0.553261
std,9.424685,0.408148,0.930969,19.066070,110.780810,0.372543,0.805827,25.926276,0.487941,1.091226,0.619256,1.919075,1.028840,0.497426
min,28.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,60.000000,0.000000,-2.600000,1.000000,3.000000,1.000000,0.000000
25%,47.000000,1.000000,3.000000,120.000000,175.000000,0.000000,0.000000,120.000000,0.000000,0.000000,1.000000,3.000000,1.000000,0.000000
50%,54.000000,1.000000,4.000000,130.000000,223.000000,0.000000,0.000000,140.000000,0.000000,0.500000,2.000000,6.000000,2.000000,1.000000
75%,60.000000,1.000000,4.000000,140.000000,268.000000,0.000000,1.000000,157.000000,1.000000,1.500000,2.000000,7.000000,3.000000,1.000000
max,77.000000,1.000000,4.000000,200.000000,603.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,7.000000,4.000000,1.000000
